## Evaluation of Cityscapes-trained model

Here we only save the predictions of the Cityscapes-trained model on semantic segmentation task.

In [1]:
import yaml
from lightning import seed_everything
import torch
import numpy as np
import warnings
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import importlib

warnings.filterwarnings("ignore")

seed_everything(0, verbose=False)

device = 0

/home/sina/Projects/ElisaMLDL/outlierdrive/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
city_config_path = "../eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
trained_city_path = "../eomt_trained/eomt_cityscapes.bin"

data_path = "../data/datasets"

output_city_path = "../data/eomt_valset_predictions/cityscapes_model"

with open(city_config_path, "r") as f:
    city_config = yaml.safe_load(f)

Now repeat the same steps as in inference_visual.ipynb and load the data.

In [3]:
import sys
sys.path.append("../eomt")

In [4]:
data_module_name, class_name = city_config["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = city_config["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
).setup()

In [5]:
val_loader = data.val_dataloader()

In [6]:
batch = next(iter(val_loader))

print(type(batch))
print(len(batch))

<class 'tuple'>
2


In [7]:
for i, item in enumerate(batch):
    print(i, type(item))

0 <class 'tuple'>
1 <class 'tuple'>


In [8]:
for i, item in enumerate(batch):
    if hasattr(item, "shape"):
        print(i, item.shape)

In [9]:
print("batch length:", len(batch))

for i, part in enumerate(batch):
    print(f"\nBatch part {i}: type={type(part)}, len={len(part)}")
    
    for j, item in enumerate(part):
        print(f"  item {j}: type={type(item)}")
        
        if hasattr(item, "shape"):
            print(f"    shape={item.shape}")
        elif isinstance(item, (list, tuple)):
            print(f"    len={len(item)}")
            if len(item) > 0:
                print(f"    first element type={type(item[0])}")
                if hasattr(item[0], "shape"):
                    print(f"    first element shape={item[0].shape}")

batch length: 2

Batch part 0: type=<class 'tuple'>, len=1
  item 0: type=<class 'torchvision.tv_tensors._image.Image'>
    shape=torch.Size([3, 1024, 2048])

Batch part 1: type=<class 'tuple'>, len=1
  item 0: type=<class 'dict'>


In [10]:
# explore the dictionary

target = batch[1][0]

print(target.keys())

for k, v in target.items():
    print(k, type(v))
    if hasattr(v, "shape"):
        print(" shape:", v.shape)
    else:
        print(" value:", v)


# Notice that GT is not already [H, W]. It is stored as multiple binary masks.

dict_keys(['masks', 'labels', 'is_crowd'])
masks <class 'torchvision.tv_tensors._mask.Mask'>
 shape: torch.Size([10, 1024, 2048])
labels <class 'torch.Tensor'>
 shape: torch.Size([10])
is_crowd <class 'torch.Tensor'>
 shape: torch.Size([10])


## Load the model

In [11]:

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = city_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=data.img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = city_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}

network = network_cls(
    masked_attn_enabled=False,
    num_classes=data.num_classes,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = city_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in city_config["model"]["init_args"].items() if k != "network"}

if "stuff_classes" in city_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = city_config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=data.img_size,
        num_classes=data.num_classes,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

## Load weights

In [12]:
weights = torch.load(
    trained_city_path,
    map_location=f"cuda:{device}",
    weights_only=False
)

if "state_dict" in weights:
    weights = weights["state_dict"]

model.load_state_dict(weights, strict=False)

model.eval()

MaskClassificationSemantic(
  (network): EoMT(
    (encoder): ViT(
      (backbone): VisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
          (norm): Identity()
        )
        (pos_drop): Dropout(p=0.0, inplace=False)
        (patch_drop): Identity()
        (norm_pre): Identity()
        (blocks): Sequential(
          (0): Block(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
            (attn): Attention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (q_norm): Identity()
              (k_norm): Identity()
              (attn_drop): Dropout(p=0.0, inplace=False)
              (norm): Identity()
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (drop_path1): Identity()
            (norm

## Semantic inference

In [13]:
IGNORE_INDEX = 255

def infer_semantic(img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)

        mask_logits = F.interpolate(
            mask_logits_per_layer[-1],
            data.img_size,
            mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits,
            class_logits_per_layer[-1]
        )

        logits = model.revert_window_logits_semantic(
            crop_logits,
            origins,
            img_sizes
        )

        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy().astype(np.uint8)

    target_array = model.to_per_pixel_targets_semantic(
        [target],
        IGNORE_INDEX
    )[0].numpy().astype(np.uint8)

    return pred_array, target_array

In [14]:
img, target = data.val_dataloader().dataset[0]

In [15]:
pred_array, target_array = infer_semantic(img, target)

In [16]:
print(pred_array.shape, pred_array.dtype)
print(target_array.shape, target_array.dtype)
print(np.unique(pred_array))
print(np.unique(target_array))

(1024, 2048) uint8
(1024, 2048) uint8
[ 0  1  2  4  5  7  8  9 10 11 12 13]
[  0   1   2   4   5   7   8  10  11  13 255]


In [17]:
from pathlib import Path
from tqdm import tqdm
import numpy as np

Next code run on Colab because otherwise it will take very long time.

In [ ]:
val_dataset = data.val_dataloader().dataset

print("Validation set size:", len(val_dataset))

for idx in tqdm(range(len(val_dataset))):

    img, target = val_dataset[idx]
    pred_array, target_array = infer_semantic(img, target)
    save_name = f"{idx:04d}_pred.npy"

    np.save(
        f"{output_city_path}/{save_name}",
        pred_array
    )

print("Finished saving predictions")

After running it on Colab, I saved the predictions in data/eomt_valset_predictions/cityscapes_model